# stochastic noise, controller ladder, and implementation defects

This notebook is part of the reproducible case39 study for the Suboptimal Control paper. Heavy Monte Carlo results used in the manuscript are cached in `../results/`; the reusable simulation and certificate code is in `../src/`.

In [ ]:
from pathlib import Path
import json, csv, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT=Path('..').resolve()
RES=ROOT/'results'; FIG=ROOT/'figures'
print(ROOT)

## Noise models

Two independent nodal white-noise models are used. **Homogeneous** noise has the same injection amplitude at every node. **Activity-scaled** noise preserves independent Wiener channels but weights their amplitudes by the pre-event nodal injection activity and then normalizes the profile to a specified RMS.  The sweep varies the common RMS parameter; it is not intended as a calibrated physical forecast, but as a controlled stress parameter.

In [ ]:
df=pd.read_csv(RES/'noise_sweep.csv'); df.head()

## Noise-amplitude sweep

The controller ladder shown here is intentionally simple: fixed balancing, generator droop, and the selected weak H3 scaled-LQR controller.  The smooth SOC cost and empirical first-exit probability are both reported.  Performance and safety are kept distinct: the smooth cost is what enters the PIC lower-bound construction; crossing $90^\circ$ is a separate safety statistic.

In [ ]:
for nz in df.noise.unique():
 d=df[df.noise==nz]; print(nz); display(d.pivot(index='sigma_rms',columns='policy',values=['cost','exit_probability']))

## Implementation defects

The general performance identity lets us price a deployed controller even when its implementation is sampled, lagged, saturated, or subject to generator dropout.  These experiments use the same H3 reference feedback and the same SOC objective; they do not re-solve the optimal-control problem after each defect.

In [ ]:
impl=json.loads((RES/'implementation.json').read_text());
for k,v in impl.items(): print(k,pd.DataFrame(v))

## Numerical convergence

The reduced swing model is stiff because load buses carry small synthetic inertia.  Coarse Euler steps can therefore create spurious exits.  The production runs use the stable step regime identified in the convergence study; figures and tables state the chosen step explicitly.

In [ ]:
co=json.loads((RES/'numerical_convergence.json').read_text()); pd.DataFrame(co)

## Heavy recomputation entry points

The reusable simulator is `../src/stochastic_swing.py`.  A production rerun should increase path counts, use fixed common-random-number seeds for policy comparisons, and save raw path statistics before regenerating the cached tables.  The paper figures themselves are regenerated by `../scripts/make_figures.py`.